# Gradient Boosting:

---

## Importing Libraries:

In [1]:
import time

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

## Importing Dataset:

In [2]:
data = pd.read_csv("Cleaned Laptop Dataset.csv")

In [3]:
data.drop(columns=['Unnamed: 0'], inplace=True)

## Building Model:

In [4]:
X = data.drop(columns=['Price'])
y = data['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [5]:
preprocessing = ColumnTransformer(transformers=[
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series']) 
], remainder='passthrough')

X_train_encoded = preprocessing.fit_transform(X_train)
X_test_encoded = preprocessing.transform(X_test)

In [6]:
param = {'loss' : ['squared_error', 'absolute_error', 'huber'],
         'learning_rate' : [0.1, 0.01],
         'n_estimators' : [200, 300, 400, 500, 600],
         'max_depth' : [3, 4, 5]}

grid = GridSearchCV(
    estimator=GradientBoostingRegressor(),
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5,
)

grid.fit(X_train_encoded, y_train)

print(f"Best Parameters: {grid.best_params_}")

Best Parameters: {'learning_rate': 0.1, 'loss': 'huber', 'max_depth': 4, 'n_estimators': 400}


In [7]:
gb = GradientBoostingRegressor(loss='huber', learning_rate=0.1, n_estimators=400, max_depth=4)

start_time = time.time()
gb.fit(X_train_encoded, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 11.91s


## Results:

In [8]:
def adjusted_r2_score(y, y_pred):

    r2 = r2_score(y, y_pred)
    n = X.shape[0]
    k = X.shape[1]

    return 1 - ((1 - r2) * (n - 1) / (n - 1 - k))

In [9]:
# Results on Training Data:

y_pred_train = gb.predict(X_train_encoded)

print(f"r2 Score (Trainig Data): {r2_score(y_train, y_pred_train) * 100 :.2f}%")
print(f"\nmean absolute error(Training Data): {mean_absolute_percentage_error(y_train, y_pred_train) * 100 :.2f}%")
print(f"\nadjusted r2 Score: {adjusted_r2_score(y_train, y_pred_train) * 100 :.2f}%")

r2 Score (Trainig Data): 91.69%

mean absolute error(Training Data): 13.07%

adjusted r2 Score: 91.66%


In [10]:
# Results on Testing Data:

y_pred_test = gb.predict(X_test_encoded)

print(f"r2 Score (Testing Data): {r2_score(y_test, y_pred_test) * 100 :.2f}%")
print(f"\nmean absolute error (Testing Data): {mean_absolute_percentage_error(y_test, y_pred_test) * 100 :.2f}%")
print(f"\nadjusted r2 Score: {adjusted_r2_score(y_test, y_pred_test) * 100 :.2f}%")

r2 Score (Testing Data): 85.70%

mean absolute error (Testing Data): 16.46%

adjusted r2 Score: 85.66%


## Cross-Val-Score:

In [11]:
preprocessing = ColumnTransformer(transformers=[
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series']) 
], remainder='passthrough')

gb_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', GradientBoostingRegressor(loss='huber', learning_rate=0.1, n_estimators=400, max_depth=4))
])

In [12]:
print(f"r2 Score: {np.mean(cross_val_score(estimator=gb_pipeline, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 84.28%


In [13]:
print(f"mean absolute error: {-np.mean(cross_val_score(estimator=gb_pipeline, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

mean absolute error: 16.44%


---

# Experimentation:

- ## Transformation (target feature):

In [14]:
preprocessing = ColumnTransformer(transformers=[
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

gb_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', GradientBoostingRegressor())
])

gb_model = TransformedTargetRegressor(
    regressor=gb_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

param = {'regressor__model__n_estimators' : [200, 300, 400, 500, 600],
        'regressor__model__loss' : ['squared_error', 'absolute_error', 'huber'],
        'regressor__model__max_depth' : [3, 4, 5],
        'regressor__model__learning_rate' : [0.1, 0.01]}

grid_model = GridSearchCV(
    estimator=gb_model,
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

grid_model.fit(X_train, y_train)

print(f"Best Parameters: {grid_model.best_params_}")

Best Parameters: {'regressor__model__learning_rate': 0.1, 'regressor__model__loss': 'huber', 'regressor__model__max_depth': 4, 'regressor__model__n_estimators': 300}


In [15]:
preprocessing = ColumnTransformer(transformers=[
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

gb_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', GradientBoostingRegressor(learning_rate=0.1, loss='huber', max_depth=5, n_estimators=300))
])

gb_model = TransformedTargetRegressor(
    regressor=gb_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

print(f"r2 Score: {np.mean(cross_val_score(estimator=gb_model, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

print(f"\nmean absolute error: {-np.mean(cross_val_score(estimator=gb_model, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 83.85%

mean absolute error: 15.34%


---

# Conclusion:

- *Gradient Boosting* performs slightly lower compared to random forest with an **r2-Score = 84.28% & M.A.P.E = 16.44%**.

- After Transforming (target feature) the r2 lowers very slightly to **83.85%** but M.A.P.E improves to **15.34%**.